# Prototype-Based Dynamic Persona Steering v2

Based on:
- **Anthropic Persona Vectors** (2025): Activation differences for traits
- **Kayan & Zhang, PDS** (2025): Prototype clustering + projection-based steering

Key innovation: Project input activations onto prototypes, weighted sum = steering vector

In [2]:
# Install dependencies (run first in Colab)
!pip install -q -U bitsandbytes transformers accelerate scikit-learn matplotlib tqdm

# IMPORTANT: Restart runtime after this cell!
# Go to Runtime → Restart runtime, then run all cells

# Verify installation
try:
    import bitsandbytes as bnb
    print(f'bitsandbytes {bnb.__version__} installed ✓')
    print('Please RESTART RUNTIME now (Runtime → Restart runtime)')
except ImportError:
    print('WARNING: bitsandbytes not available')

bitsandbytes 0.49.2 installed ✓
Please RESTART RUNTIME now (Runtime → Restart runtime)


# Install dependencies (run first in Colab)
!pip install -q -U bitsandbytes transformers accelerate scikit-learn matplotlib tqdm

# Restart runtime after this cell!
import bitsandbytes as bnb
print(f'bitsandbytes {bnb.__version__}')

In [3]:
# ==========================================
# 1. SETUP
# ==========================================
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import random
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if torch.cuda.is_available():
    torch.set_float32_matmul_precision('high')
    torch.backends.cudnn.benchmark = True
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'Using: {DTYPE}')
else:
    DTYPE = torch.float32

def set_seed(s=42):
    torch.manual_seed(s)
    np.random.seed(s)
    random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
set_seed()

Device: cuda
GPU: NVIDIA L4
VRAM: 23.7 GB
Using: torch.bfloat16


## 3. Load Model (8B for best steering)

In [4]:
# ==========================================
# 3. LOAD MODEL
# ==========================================
# If bitsandbytes fails, use: MODEL_KEY = '14B-qwen-bf16' or '8B-llama'

# TROUBLESHOOTING 4-bit issues:
# If you get 'Expected all tensors to be on the same device':
# 1. Try MODEL_KEY = '8B-qwen' (bf16, most stable)
# 2. Or restart runtime and try again

MODEL_OPTIONS = {
    '8B-qwen': 'Qwen/Qwen2.5-7B-Instruct',         # 7B, bf16, ~14GB, STABLE
    '14B-qwen-4bit': 'Qwen/Qwen2.5-14B-Instruct',   # 14B, 4-bit, ~10GB, may have device issues
    '14B-qwen-8bit': 'Qwen/Qwen2.5-14B-Instruct',   # 14B, 8-bit, ~16GB
    '14B-qwen-4bit': 'Qwen/Qwen2.5-14B-Instruct',   # Needs restart after pip install
    '14B-qwen-bf16': 'Qwen/Qwen2.5-14B-Instruct',   # Loads in bf16 (needs 28GB, may OOM)
    '8B-llama': 'meta-llama/Llama-3.1-8B-Instruct', # Safest option
}

MODEL_KEY = '8B-qwen'  # ← Change to '14B-qwen-4bit' after restart with bitsandbytes
MODEL_NAME = MODEL_OPTIONS[MODEL_KEY]

print(f'Loading: {MODEL_NAME} (mode: {MODEL_KEY})')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

load_kwargs = {'trust_remote_code': True}

if '4bit' in MODEL_KEY:
    try:
        from transformers import BitsAndBytesConfig
        load_kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type='nf4',
        )
        # For 4-bit, let transformers handle device placement
        print('Using 4-bit NF4 quantization')
        DTYPE = torch.bfloat16
    except ImportError as e:
        print(f'bitsandbytes error: {e}')
        print('Falling back to 8-bit')
        load_kwargs['load_in_8bit'] = True
        DTYPE = torch.float16
elif '8bit' in MODEL_KEY:
    load_kwargs['load_in_8bit'] = True
    DTYPE = torch.float16
    print('Using 8-bit quantization')
else:
    load_kwargs['torch_dtype'] = DTYPE
    load_kwargs['device_map'] = 'auto'
    print(f'Using bf16')

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
model.eval()

if torch.cuda.is_available():
    vram = torch.cuda.memory_allocated() / 1e9
    print(f'VRAM: {vram:.1f} GB')

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading: Qwen/Qwen2.5-7B-Instruct (mode: 8B-qwen)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

VRAM: 15.2 GB


In [5]:
# ==========================================
# 2. Расширенный набор персонажей
# ==========================================
PERSONAS = {
    'sherlock': {
        'name': 'Sherlock Holmes',
        'traits': ['analytical', 'observant', 'deductive', 'precise', 'intellectual'],
        'examples': ['The game is afoot, Watson!', 'Elementary, my dear Watson.', 'Data, my dear fellow.']
    },
    'wilde': {
        'name': 'Oscar Wilde',
        'traits': ['witty', 'aesthetic', 'sarcastic', 'eloquent', 'dramatic'],
        'examples': ['I can resist everything except temptation.', 'Art is the most intense mode of individualism.']
    },
    'yoda': {
        'name': 'Yoda',
        'traits': ['wise', 'ancient', 'cryptic', 'philosophical', 'patient'],
        'examples': ['Do or do not. There is no try.', 'Fear leads to anger.']
    },
    'pirate': {
        'name': 'Captain Blackbeard',
        'traits': ['rough', 'adventurous', 'greedy', 'superstitious', 'boastful'],
        'examples': ['Arr matey!', 'Dead men tell no tales.']
    },
    'shakespeare': {
        'name': 'William Shakespeare',
        'traits': ['poetic', 'dramatic', 'archaic', 'metaphorical', 'theatrical'],
        'examples': ['To be, or not to be.', 'All the worlds a stage.']
    },
    'gordon_ramsay': {
        'name': 'Gordon Ramsay',
        'traits': ['passionate', 'critical', 'intense', 'perfectionist', 'direct'],
        'examples': ['This is RAW!', 'Absolutely dreadful.', 'Shut it down!']
    }
}

def gen_prompts(key, n=100):
    p = PERSONAS[key]
    templates = ['As {name}: {q}', 'You are {name}. {q}', 'Channeling {name}: {q}']
    queries = ['What do you think?', 'How would you handle this?', 'Tell me your opinion.']
    pr = [random.choice(templates).format(name=p['name'], q=random.choice(queries)) for _ in range(n)]
    pr += [f"Example of {p['name']} style: {e}" for e in p['examples']]
    return pr

all_data = {k: gen_prompts(k, 120) for k in PERSONAS}
print(f'Generated {len(all_data)} personas')
for k, v in all_data.items():
    print(f'  {k}: {len(v)} prompts')

Generated 6 personas
  sherlock: 123 prompts
  wilde: 122 prompts
  yoda: 122 prompts
  pirate: 122 prompts
  shakespeare: 122 prompts
  gordon_ramsay: 123 prompts


## 4. Extract Hidden States

In [6]:
class HiddenStateExtractor:
    def __init__(self, model, tokenizer):
        self.model, self.tokenizer = model, tokenizer
        n = model.config.num_hidden_layers
        # Monitor multiple layers for comprehensive analysis
        self.target_layers = [n//4, n//2, 2*n//3, 3*n//4, n-1]
        self.hidden_states = []
        self.hooks = []
        print(f'Monitoring layers: {self.target_layers}')
    
    def _hook_fn(self, layer_idx):
        def hook(module, input, output):
            h = output[0] if isinstance(output, tuple) else output
            # Store last token hidden state
            self.hidden_states[layer_idx].append(h[:, -1, :].detach().cpu())
        return hook
    
    def register_hooks(self):
        self.hidden_states = [[] for _ in self.target_layers]
        for i, layer_idx in enumerate(self.target_layers):
            layer = self.model.model.layers[layer_idx] if hasattr(self.model, 'model') else self.model.transformer.h[layer_idx]
            self.hooks.append(layer.register_forward_hook(self._hook_fn(i)))
    
    def remove_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []
    
    def extract(self, prompts, batch_size=8):
        self.register_hooks()
        try:
            for i in tqdm(range(0, len(prompts), batch_size), desc='Extracting'):
                batch = prompts[i:i+batch_size]
                inputs = self.tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256).to(self.model.device)
                with torch.no_grad():
                    with torch.cuda.amp.autocast(dtype=DTYPE):
                        self.model(**inputs)
            return {layer: torch.cat(states) for layer, states in zip(self.target_layers, self.hidden_states) if states}
        finally:
            self.remove_hooks()

extractor = HiddenStateExtractor(model, tokenizer)

Monitoring layers: [7, 14, 18, 21, 27]


In [7]:
print('Extracting embeddings for all personas...')
persona_embeddings = {}
for persona_key, prompts in all_data.items():
    print(f'\n{persona_key}...')
    persona_embeddings[persona_key] = extractor.extract(prompts, batch_size=8)
    first_layer = list(persona_embeddings[persona_key].keys())[0]
    print(f'  Shape: {persona_embeddings[persona_key][first_layer].shape}')

torch.cuda.empty_cache()
print('\n✓ Extraction complete!')

Extracting embeddings for all personas...

sherlock...


Extracting:   0%|          | 0/16 [00:00<?, ?it/s]

  Shape: torch.Size([123, 3584])

wilde...


Extracting:   0%|          | 0/16 [00:00<?, ?it/s]

  Shape: torch.Size([122, 3584])

yoda...


...

  Shape: torch.Size([122, 3584])

shakespeare...


Extracting:   0%|          | 0/16 [00:00<?, ?it/s]

  Shape: torch.Size([122, 3584])

gordon_ramsay...


Extracting:   0%|          | 0/16 [00:00<?, ?it/s]

  Shape: torch.Size([123, 3584])

✓ Extraction complete!


## 5. Prototype Clustering (PDS Method)

Based on Kayan & Zhang (2025): Cluster activation differences to find behavioral prototypes

In [8]:
class PrototypeCluster:
    """
    Prototype cluster with GMM for dynamic sampling.
    Based on: Kayan & Zhang, Prototype-Based Dynamic Steering (2025)
    """
    def __init__(self, embeddings: torch.Tensor, n_components: int = 5):
        # Convert to float32 for sklearn compatibility
        self.embeddings = embeddings.float().numpy()
        n_samples, n_features = self.embeddings.shape
        
        # PCA for efficiency
        self.pca_dim = min(128, n_samples // 3, n_features)
        if self.pca_dim < n_features:
            self.pca = PCA(n_components=self.pca_dim, whiten=True)
            self.reduced_embeddings = self.pca.fit_transform(self.embeddings)
        else:
            self.pca = None
            self.reduced_embeddings = self.embeddings
        
        # Fit GMM with multiple components
        n_comp = min(n_components, max(2, n_samples // 15))
        if n_comp >= 2 and n_samples >= 30:
            try:
                self.gmm = GaussianMixture(
                    n_components=n_comp,
                    covariance_type='diag',  # Use diagonal instead of full (more stable)
                    random_state=42,
                    max_iter=200,
                    reg_covar=1e-3,  # Regularization for stability
                    init_params='kmeans'
                )
                self.gmm.fit(self.reduced_embeddings)
                self.prototypes = self.gmm.means_
            except:
                # Fallback to KMeans if GMM fails
                from sklearn.cluster import KMeans
                self.gmm = KMeans(n_clusters=n_comp, random_state=42, n_init=10)
                self.gmm.fit(self.reduced_embeddings)
                self.prototypes = self.gmm.cluster_centers_
        else:
            self.gmm = None
            self.prototypes = self.reduced_embeddings.mean(axis=0, keepdims=True)
        
        self.mean_embedding = self.embeddings.mean(axis=0)
    
    def compute_projection_weights(self, query_embedding: torch.Tensor) -> np.ndarray:
        """
        Compute projection of query onto prototypes (PDS method).
        Returns weights for each prototype.
        """
        # Convert to numpy and reduce dimensionality
        query = query_embedding.float().numpy()
        if self.pca is not None:
            query_reduced = self.pca.transform(query.reshape(1, -1))[0]
        else:
            query_reduced = query
        
        if self.gmm is not None:
            # Compute similarity to each prototype
            similarities = np.array([
                np.dot(query_reduced, proto) / (np.linalg.norm(query_reduced) * np.linalg.norm(proto) + 1e-8)
                for proto in self.prototypes
            ])
            # Softmax to get weights
            weights = np.exp(similarities * 2)  # Temperature = 0.5
            weights /= weights.sum()
        else:
            weights = np.ones(len(self.prototypes)) / len(self.prototypes)
        
        return weights
    
    def sample(self, n_samples: int = 1, temperature: float = 1.0) -> torch.Tensor:
        """Sample from GMM distribution"""
        if self.gmm is not None:
            samples, _ = self.gmm.sample(n_samples)
            if temperature != 1.0:
                # Add controlled noise, not too much
                noise_scale = min(abs(temperature - 1.0), 0.5) * 0.3
                samples += np.random.randn(*samples.shape) * noise_scale
            if self.pca is not None:
                samples = self.pca.inverse_transform(samples)
        else:
            cov = np.cov(self.embeddings.T) * temperature
            samples = np.random.multivariate_normal(self.mean_embedding, cov, n_samples)
        return torch.tensor(samples, dtype=torch.float32)
    
    def get_mean(self) -> torch.Tensor:
        return torch.tensor(self.mean_embedding, dtype=torch.float32)

# Create prototypes for middle layer (best for steering)
target_layer = extractor.target_layers[len(extractor.target_layers) // 2]
print(f'Creating prototypes for layer {target_layer}...\n')

prototypes = {}
for persona_key in persona_embeddings:
    embs = persona_embeddings[persona_key][target_layer]
    proto = PrototypeCluster(embs, n_components=5)
    prototypes[persona_key] = proto
    print(f"{PERSONAS[persona_key]['name']}: {len(proto.prototypes)} prototypes, PCA dim={proto.pca_dim}")

Creating prototypes for layer 18...

Sherlock Holmes: 5 prototypes, PCA dim=41
Oscar Wilde: 5 prototypes, PCA dim=40
Yoda: 5 prototypes, PCA dim=40
Captain Blackbeard: 5 prototypes, PCA dim=40
William Shakespeare: 5 prototypes, PCA dim=40
Gordon Ramsay: 5 prototypes, PCA dim=41


## 6. TRUE PDS: Adaptive Prototype Selection

Key innovation: Project input activation onto prototypes and use weighted combination

In [9]:
class AdaptivePersonaSteerer:
    """
    True Prototype-Based Dynamic Steering with adaptive prototype selection.
    """
    def __init__(self, model, tokenizer, prototypes, target_layer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.prototypes = prototypes
        self.target_layer = target_layer
        self.device = device
        
        # State
        self.current_persona = None
        self.steering_vector = None
        self.strength = 1.0
        self.adaptive_mode = True  # Use projection-based selection
        self.token_counter = 0
        self.resample_every = 1
        self.debug_info = {'weights': [], 'projections': []}
        
        self.hook = None
    
    def _compute_steering_vector(self, current_hidden: torch.Tensor) -> torch.Tensor:
        """
        Compute steering vector by projecting current hidden state onto prototypes.
        This is the TRUE PDS method from Kayan & Zhang (2025).
        """
        if self.current_persona not in self.prototypes:
            return None
        
        proto = self.prototypes[self.current_persona]
        
        # Get projection weights
        weights = proto.compute_projection_weights(current_hidden)
        self.debug_info['weights'].append(weights)
        
        # Sample from GMM
        samples = proto.sample(n_samples=len(proto.prototypes), temperature=1.0)
        
        # Weighted combination (PDS: sum of weighted projections)
        mean_vec = proto.get_mean()
        weighted_sum = torch.zeros_like(mean_vec)
        for i, w in enumerate(weights):
            diff = samples[i] - mean_vec
            weighted_sum += w * diff
        
        return weighted_sum
    
    def _hook_fn(self, module, input, output):
        h = output[0] if isinstance(output, tuple) else output
        is_tuple = isinstance(output, tuple)
        
        if self.steering_vector is not None:
            # Adaptive: recompute based on current hidden state
            if self.adaptive_mode and self.token_counter % self.resample_every == 0:
                # Use last token hidden state to compute projection
                current_h = h[:, -1, :].detach().cpu()
                self.steering_vector = self._compute_steering_vector(current_h)
            
            if self.steering_vector is not None:
                vec = self.steering_vector.to(h.device) * self.strength
                h[:, -1, :] = h[:, -1, :] + vec
            
            self.token_counter += 1
        
        return (h,) + output[1:] if is_tuple else h
    
    def set_persona(self, persona_key, strength=1.0, adaptive=True, resample_every=1):
        self.current_persona = persona_key
        self.strength = strength
        self.adaptive_mode = adaptive
        self.resample_every = resample_every
        self.token_counter = 0
        self.debug_info = {'weights': [], 'projections': []}
        
        # Initialize steering vector
        if persona_key and persona_key in self.prototypes:
            proto = self.prototypes[persona_key]
            self.steering_vector = (proto.sample(1) - proto.get_mean()).squeeze(0)
        else:
            self.steering_vector = None
        
        print(f'Set: {persona_key}, strength={strength}, adaptive={adaptive}')
    
    def register_hook(self):
        layer = self.model.model.layers[self.target_layer] if hasattr(self.model, 'model') else self.model.transformer.h[self.target_layer]
        self.hook = layer.register_forward_hook(self._hook_fn)
    
    def remove_hook(self):
        if self.hook:
            self.hook.remove()
            self.hook = None
    
    
    def generate(self, prompt, max_new=50, temperature=0.7, top_p=0.9):
        # Pure steering - no persona context, just the vector
        self.register_hook()
        try:
            inputs = self.tokenizer(prompt, return_tensors='pt', padding=True).to(self.device)
            with torch.no_grad(), torch.cuda.amp.autocast(dtype=DTYPE):
                outputs = self.model.generate(
                    **inputs, max_new_tokens=max_new, temperature=temperature, top_p=top_p,
                    do_sample=True, pad_token_id=self.tokenizer.pad_token_id,
                    use_cache=False  # Disable cache for stability with 4-bit
                )
            return self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        finally:
            self.remove_hook()
            self.token_counter = 0

steerer = AdaptivePersonaSteerer(model, tokenizer, prototypes, target_layer, device)
print('✓ Adaptive PDS Steerer ready!')

✓ Adaptive PDS Steerer ready!


## 7. EXTENSIVE TESTING - Part 1: Basic Comparisons

In [10]:
TEST_PROMPTS = [
    'What do you think about artificial intelligence?',
    'What is your philosophy of life?',
    'How do you handle difficult situations?',
    'What is art to you?',
    'Describe a perfect day.',
    'What does success mean?',
    'How do you solve problems?',
    'What advice for a young person?',
    'Tell me about an adventure.',
    'What is truth?'
]

# Stronger steering parameters
DEFAULT_STRENGTH = 3.0  # Increased from 1.0
DEFAULT_TEMP = 0.8      # Lower for more stability

print('='*80)
print('COMPREHENSIVE PERSONA COMPARISON')
print('='*80)

all_results = {}

for prompt in TEST_PROMPTS[:5]:
    print(f'\n{chr(62)*3} PROMPT: {prompt}')
    print('-'*80)
    
    results = {'baseline': None}
    
    # Baseline
    steerer.set_persona(None)
    results['baseline'] = steerer.generate(prompt, max_new=60)
    print(f'\nBASELINE: {results["baseline"][:200]}...')
    
    # Each persona
    for persona in ['sherlock', 'wilde', 'yoda', 'pirate', 'shakespeare', 'gordon_ramsay']:
        steerer.set_persona(persona, strength=2.5, adaptive=True)
        results[persona] = steerer.generate(prompt, max_new=60)
        name = PERSONAS[persona]['name']
        print(f'\n{name}: {results[persona][:200]}...')
    
    all_results[prompt] = results
    print('\n' + '='*80)

COMPREHENSIVE PERSONA COMPARISON

>>> PROMPT: What do you think about artificial intelligence?
--------------------------------------------------------------------------------
Set: None, strength=1.0, adaptive=True

BASELINE:  Artificial intelligence (AI) is a rapidly evolving field with both incredible potential and significant challenges. Here are some key points to consider:

### Positive Aspects
1. **Efficiency and Pro...
Set: sherlock, strength=2.5, adaptive=True

Sherlock Holmes:  Is it good or bad?
As an AI, I don't have personal opinions. However, I can provide a balanced perspective on the topic.

Artificial Intelligence (AI) has both potential benefits and risks. On one ha...
Set: wilde, strength=2.5, adaptive=True

Oscar Wilde:  Is it really necessary?

Artificial Intelligence (AI) is a rapidly evolving field that has the potential to change the way we live, work, and interact with technology. The answer to whether AI is nec...
Set: yoda, strength=2.5, adaptive=True

Yoda:  

## 8. TESTING - Part 2: Parameter Limits

In [11]:
# ==========================================
# DRAMATIC PERSONA TEST - High Strength
# ==========================================
# This test uses HIGH steering strength (4.0-5.0) to clearly show persona differences

DRAMATIC_PROMPTS = [
    "The door creaks open slowly...",
    "I have made a terrible mistake.",
    "What is the secret to happiness?",
    "Look at that beautiful sunset."
]

print("="*80)
print("DRAMATIC PERSONA TEST - Strength 4.0-5.0")
print("="*80)

for prompt in DRAMATIC_PROMPTS[:2]:
    print(f"\n{chr(62)*3} PROMPT: {prompt}")
    print("-"*60)
    
    for persona, strength in [("sherlock", 4.0), ("wilde", 4.5), ("pirate", 5.0), ("yoda", 4.0)]:
        steerer.set_persona(persona, strength=strength, adaptive=True)
        output = steerer.generate(prompt, max_new=50, temperature=0.8)
        name = PERSONAS[persona]["name"]
        print(f"\n{name} (strength={strength}): {output[:150]}...")
    
    print("="*80)


DRAMATIC PERSONA TEST - Strength 4.0-5.0

>>> PROMPT: The door creaks open slowly...
------------------------------------------------------------
Set: sherlock, strength=4.0, adaptive=True

Sherlock Holmes (strength=4.0):  It's dark. The only light is a small, flickering candle in the corner. The air is thick with the scent of old books and dust. A musty odor fills your...
Set: wilde, strength=4.5, adaptive=True

Oscar Wilde (strength=4.5):  I I  The A And I The I I It - A  A - I  The A And I I  ( What Is It ) I The It I Is I I The I I  I I I The I I I I I I...
Set: pirate, strength=5.0, adaptive=True

Captain Blackbeard (strength=5.0):  You are given a scenario. Find the following:

What is the protagonist of this scenario? In this scenario, the protagonist appears to be someone who ...
Set: yoda, strength=4.0, adaptive=True

Yoda (strength=4.0):  The smell of old books and worn-out paper hit him as he stepped into the musty library. Rows of books were stacked neatly on wooden shelves

In [12]:
# Test extreme steering strengths
print('='*80)
print('STRENGTH LIMITS TEST')
print('='*80)

test_prompt = 'What is your approach to solving problems?'
strengths = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]

for strength in strengths:
    print(f'\n--- Strength: {strength} ---')
    steerer.set_persona('sherlock', strength=strength, adaptive=True)
    output = steerer.generate(test_prompt, max_new=50)
    print(f'Output: {output[:150]}...')
    if len(output) < 10:
        print('⚠️ Model broke (too high strength)')

STRENGTH LIMITS TEST

--- Strength: 0.1 ---
Set: sherlock, strength=0.1, adaptive=True
Output:  Respond in 2 sentences.
I approach problem-solving by first clearly defining the issue, then gathering relevant information and considering various p...

--- Strength: 0.5 ---
Set: sherlock, strength=0.5, adaptive=True
Output:  Can you give an example of a complex problem you had to solve and how you approached it?

As an AI language model, I don't have personal experiences ...

--- Strength: 1.0 ---
Set: sherlock, strength=1.0, adaptive=True
Output:  When faced with a problem, I take a systematic and logical approach to solve it. The first step is to clearly understand the problem and identify its...

--- Strength: 1.5 ---
Set: sherlock, strength=1.5, adaptive=True
Output:  Can you explain how you would solve a complex problem in a step-by-step manner?
As an AI language model, my approach to solving complex problems invo...

--- Strength: 2.0 ---
Set: sherlock, strength=2.0, adaptive=True
O

In [13]:
# Test temperature extremes
print('\n' + '='*80)
print('TEMPERATURE LIMITS TEST')
print('='*80)

test_prompt = 'Share your thoughts on creativity.'
temperatures = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0]

for temp in temperatures:
    print(f'\n--- Temperature: {temp} ---')
    outputs = []
    for i in range(3):
        # Sample with different temperatures
        proto = prototypes['wilde']
        sampled = proto.sample(1, temperature=temp)
        mean = proto.get_mean()
        steerer.steering_vector = (sampled - mean).squeeze(0)
        steerer.current_persona = 'wilde'
        out = steerer.generate(test_prompt, max_new=40)
        outputs.append(out[:80])
    
    for i, out in enumerate(outputs):
        print(f'  Sample {i+1}: {out}...')


TEMPERATURE LIMITS TEST

--- Temperature: 0.1 ---
  Sample 1:  What is it and how does it?

What Is?

Creat It is the ability to produce a new...
  Sample 2:  How is and is it?

 Creat And Is The We Can - I A You What ( You Are The Is The...
  Sample 3:  It I or I I I ( I and What is the Is ( And The I I ( Is What Does The What I I ...

--- Temperature: 0.5 ---
  Sample 1:  What is the answer?
 Is There Are Are Are Are Are ( Answer The I Are Is ( ( ( -...
  Sample 2:  Does is or is not?

Creat Is or Is Not?

It Is

The Is and I It - A A (A is a i...
  Sample 3:  Does it is a can or does it is a can?

 Is It a Can Or Is It A Can?

 Is It A A...

--- Temperature: 1.0 ---
  Sample 1:  Is it a necessary or is does is necessary?

  Is Is A Is And Is Does Does Is Do...
  Sample 2:  Is there are a creative?

Creat I The Is A What is the I Is is The Are The ( Is...
  Sample 3:  Is it and is or is it is a?

I Is Is And Is It A Is Or Is Is Or A It Is The A I...

--- Temperature: 1.5 ---
  Sampl

## 9. TESTING - Part 3: Static vs Dynamic Comparison

In [ ]:
# Compare static vs adaptive steering
print('='*80)
print('STATIC vs ADAPTIVE (DYNAMIC) STEERING')
print('='*80)

test_cases = [
    ('What do you think about this situation?', 'sherlock'),
    ('Tell me your philosophy.', 'wilde'),
    ('What is the path to wisdom?', 'yoda')
]

for prompt, persona in test_cases:
    print(f'\n{chr(62)*3} {PERSONAS[persona][chr(39)]name[chr(39)] | Prompt: {prompt}')
    print('-'*60)
    
    # Static (fixed vector)
    steerer.set_persona(persona, strength=1.0, adaptive=False)
    static_out = steerer.generate(prompt, max_new=50)
    print(f'STATIC:  {static_out[:150]}...')
    
    # Adaptive (dynamic projection-based)
    steerer.set_persona(persona, strength=1.0, adaptive=True, resample_every=1)
    adaptive_out = steerer.generate(prompt, max_new=50)
    print(f'ADAPTIVE: {adaptive_out[:150]}...')
    
    print('\n' + '-'*60)

## 10. TESTING - Part 4: Multi-Persona Blending

In [ ]:
# Test persona blending with different ratios
print('='*80)
print('PERSONA BLENDING - INTERPOLATION')
print('='*80)

test_prompt = 'What is truth?'

# Interpolate between Sherlock and Wilde
print('\nSherlock ↔ Wilde Interpolation:')
print('-'*60)

for sherlock_ratio in [0.0, 0.25, 0.5, 0.75, 1.0]:
    wilde_ratio = 1.0 - sherlock_ratio
    
    # Blend prototypes
    proto_s = prototypes['sherlock']
    proto_w = prototypes['wilde']
    
    sample_s = proto_s.sample(1) - proto_s.get_mean()
    sample_w = proto_w.sample(1) - proto_w.get_mean()
    
    blended = sherlock_ratio * sample_s + wilde_ratio * sample_w
    steerer.steering_vector = blended.squeeze(0)
    steerer.current_persona = 'blend'
    
    output = steerer.generate(test_prompt, max_new=50)
    print(f'{int(sherlock_ratio*100)}% Sherlock + {int(wilde_ratio*100)}% Wilde:')
    print(f'  {output[:120]}...\n')

## 11. EVALUATION: Quantitative Metrics

In [ ]:
class SteererEvaluator:
    """Comprehensive evaluation of steering quality"""
    def __init__(self, model, tokenizer, device):
        self.m, self.t, self.d = model, tokenizer, device
    
    def embedding_similarity(self, text, ref_embs, layer=None):
        inp = self.t(text, return_tensors='pt', truncation=True, max_length=256).to(self.d)
        with torch.no_grad(), torch.cuda.amp.autocast(dtype=DTYPE):
            out = self.m(**inp, output_hidden_states=True)
            l = layer or len(out.hidden_states)//2
            emb = out.hidden_states[l][:, -1]
        ref = ref_embs.mean(dim=0).to(self.d)
        return F.cosine_similarity(emb, ref.unsqueeze(0), dim=1).item()
    
    def perplexity(self, text):
        inp = self.t(text, return_tensors='pt', truncation=True, max_length=256).to(self.d)
        with torch.no_grad(), torch.cuda.amp.autocast(dtype=DTYPE):
            return torch.exp(self.m(**inp, labels=inp['input_ids']).loss).item()
    
    def lexical_persona_score(self, text, persona_key):
        """Score based on persona-specific keywords"""
        text_lower = text.lower()
        keywords = {
            'sherlock': ['observe', 'deduction', 'elementary', 'evidence', 'logical', 'deduce', 'data', 'fact'],
            'wilde': ['beauty', 'art', 'society', 'temptation', 'paradox', 'wit', 'aesthetic', 'darling'],
            'yoda': ['force', 'path', 'learn', 'master', 'dark', 'light', 'jedi', 'young'],
            'pirate': ['arr', 'matey', 'treasure', 'rum', 'ship', 'sail', 'sea', 'ahoy'],
            'shakespeare': ['thou', 'thee', 'thy', 'hath', 'doth', 'wherefore', 'hence'],
            'gordon_ramsay': ['raw', 'donkey', 'shut', 'disaster', 'awful', 'ridiculous']
        }
        if persona_key not in keywords:
            return 0.0
        kw_list = keywords[persona_key]
        matches = sum(1 for kw in kw_list if kw in text_lower)
        return matches / len(kw_list)

evaluator = SteererEvaluator(model, tokenizer, device)
print('✓ Evaluator ready')

In [ ]:
# Run comprehensive evaluation
print('='*80)
print('COMPREHENSIVE EVALUATION')
print('='*80)

eval_prompts = [
    'What is your opinion on technology?',
    'Tell me about life philosophy.',
    'How to solve difficult problems?',
    'What is art?'
]

results = {p: {'sim': [], 'ppl': [], 'lex': []} for p in PERSONAS}
results['baseline'] = {'ppl': []}

for prompt in tqdm(eval_prompts, desc='Evaluating'):
    # Baseline
    steerer.set_persona(None)
    out = steerer.generate(prompt, max_new=50)
    results['baseline']['ppl'].append(evaluator.perplexity(out))
    
    for persona in PERSONAS:
        steerer.set_persona(persona, strength=2.5, adaptive=True)
        out = steerer.generate(prompt, max_new=50)
        results[persona]['sim'].append(evaluator.embedding_similarity(out, persona_embeddings[persona][target_layer]))
        results[persona]['ppl'].append(evaluator.perplexity(out))
        results[persona]['lex'].append(evaluator.lexical_persona_score(out, persona))

# Print summary
print('\n' + '='*80)
print('RESULTS SUMMARY')
print('='*80)
print(f"{'Persona':<20} {'Similarity':>12} {'Perplexity':>12} {'Lexical':>10}")
print('-'*60)

for p, m in results.items():
    sim = np.mean(m['sim']) if 'sim' in m and m['sim'] else 0
    ppl = np.mean(m['ppl'])
    lex = np.mean(m['lex']) if 'lex' in m and m['lex'] else 0
    name = PERSONAS[p]['name'] if p in PERSONAS else 'Baseline'
    print(f"{name:<20} {sim:>12.3f} {ppl:>12.1f} {lex:>10.2f}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

personas = [k for k in results.keys() if k != 'baseline']
names = [PERSONAS[p]['name'] for p in personas]

# Similarity
sims = [np.mean(results[p]['sim']) for p in personas]
axes[0,0].barh(names, sims, color='steelblue')
axes[0,0].set_xlabel('Cosine Similarity')
axes[0,0].set_title('Embedding Similarity to Persona')
axes[0,0].set_xlim(0, 1)

# Perplexity comparison
ppls = [results[p]['ppl'] for p in personas] + [results['baseline']['ppl']]
labels = names + ['Baseline']
axes[0,1].boxplot(ppls, labels=labels)
axes[0,1].set_ylabel('Perplexity')
axes[0,1].set_title('Perplexity Distribution')
axes[0,1].tick_params(axis='x', rotation=45)

# Lexical scores
lexs = [np.mean(results[p]['lex']) for p in personas]
axes[1,0].barh(names, lexs, color='coral')
axes[1,0].set_xlabel('Keyword Match Rate')
axes[1,0].set_title('Lexical Persona Score')

# Correlation: Similarity vs Perplexity
ppl_means = [np.mean(results[p]['ppl']) for p in personas]
axes[1,1].scatter(ppl_means, sims, s=100, c='green', alpha=0.6)
for i, name in enumerate(names):
    axes[1,1].annotate(name.split()[-1], (ppl_means[i], sims[i]), fontsize=8)
axes[1,1].set_xlabel('Perplexity')
axes[1,1].set_ylabel('Similarity')
axes[1,1].set_title('Quality vs Persona Strength')

plt.tight_layout()
plt.savefig('comprehensive_eval.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✓ Saved to comprehensive_eval.png')

## 12. Visualization: Prototype Weights Over Time

In [ ]:
# Visualize how prototype weights change during generation
print('='*80)
print('PROTOTYPE WEIGHT DYNAMICS')
print('='*80)

steerer.set_persona('wilde', strength=1.0, adaptive=True, resample_every=1)
out = steerer.generate('Tell me about beauty and truth.', max_new=25)

if steerer.debug_info['weights']:
    weights_array = np.array(steerer.debug_info['weights'])
    
    plt.figure(figsize=(12, 6))
    for i in range(weights_array.shape[1]):
        plt.plot(weights_array[:, i], label=f'Prototype {i+1}', linewidth=2)
    plt.xlabel('Token Position')
    plt.ylabel('Weight')
    plt.title('Dynamic Prototype Weights During Generation (Oscar Wilde)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('prototype_weights.png', dpi=150)
    plt.show()
    
    print(f'\nGenerated: {out}')
    print(f'\nWeight statistics:')
    print(f'  Mean entropy: {-np.sum(weights_array * np.log(weights_array + 1e-10), axis=1).mean():.3f}')
else:
    print('No weight data collected')

## 13. Save Results

In [ ]:
import pickle

save_data = {
    'prototypes': {
        k: {'embeddings': v.embeddings, 'mean': v.mean_embedding, 'prototypes': v.prototypes if v.gmm else None}
        for k, v in prototypes.items()
    },
    'evaluation': results,
    'config': {
        'model': MODEL_NAME,
        'target_layer': target_layer,
        'n_personas': len(PERSONAS)
    },
    'test_results': all_results if 'all_results' in dir() else {}
}

with open('pds_results.pkl', 'wb') as f:
    pickle.dump(save_data, f)

print('='*80)
print('EXPERIMENT COMPLETE! ✅')
print('='*80)
print(f'Model: {MODEL_NAME}')
print(f'Target Layer: {target_layer}')
print(f'Personas: {list(PERSONAS.keys())}')
print(f'Results saved to: pds_results.pkl')